# GA-STOMP on CWRU bearing data

Builds the CUDA binary, runs the GA on four CWRU drive-end signals (12 kHz, 1797 rpm),
and writes everything to `/kaggle/working/results/` (CSVs, logs, figures, `RUN_STATUS.txt`).

Run it from the Kaggle UI, or let the GitHub Actions workflow push it with `kaggle kernels push`
(the workflow replaces `GIT_REF` below with the commit SHA it is testing).

Expected windows from bearing geometry at 1797 rpm (29.95 Hz shaft):
BPFI = 5.415 x 29.95 = 162.2 Hz -> 12000 / 162.2 = **74 samples**;
BPFO = 3.585 x 29.95 = 107.4 Hz -> 12000 / 107.4 = **112 samples**.

In [ ]:
# ---- Settings ---------------------------------------------------------------
GIT_REF  = "v6-spacing-regularity"   # branch, tag or SHA; CI overwrites this line
REPO_URL = "https://github.com/HoaPNG520/Motif-finding-GA-STOMP.git"

N_SAMPLES   = 6000      # samples taken from the start of each signal
CWRU_SLUG   = "sufian79/cwru-mat-full-dataset"
DATASETS = {            # name: (mat file, expected window or None)
    "IR007":  ("105.mat", 74),
    "IR014":  ("106.mat", 74),
    "OR007":  ("130.mat", 112),
    "Normal": ("100.mat", None),
}

CONFIG_TEMPLATE = """
m_min=20
m_max=200
ez_min=0.25
ez_max=1.0
k_min=2
k_max=20
population=20
generations=20
tournament_k=3
mutation_rate=0.30
elite_count=2
approx_frac=0.5
n_seeds=3
verbose=1
sampling_rate=12000
fault_freq_min_hz=50
fault_freq_max_hz=300
input_path={input_path}
output_dir={output_dir}
"""

import os, subprocess, shutil, glob
SRC_DIR  = "/tmp/GA_STOMP"           # repo checkout (kept out of /kaggle/working)
DATA_DIR = "/tmp/ga_data"
OUT_DIR  = "/kaggle/working/results"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
status = {}                          # step -> "OK" / "FAIL: reason"

def sh(cmd, **kw):
    """Run a command, echo it, never raise (so results are always saved)."""
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), text=True,
                       capture_output=True, **kw)
    if r.stdout: print(r.stdout[-4000:])
    if r.stderr: print("STDERR:", r.stderr[-4000:])
    return r

In [ ]:
# ---- Kaggle API credentials (optional) --------------------------------------
# Only needed if the CWRU dataset is not attached to this notebook and has to be
# downloaded. Add a Kaggle secret named KAGGLE_API_TOKEN (and optionally
# KAGGLE_USERNAME) under Add-ons -> Secrets. Nothing is written to disk.
have_api = False
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME"):
        try:
            val = secrets.get_secret(name)
            if val:
                os.environ[name] = val
                print(f"[secrets] {name} loaded")
        except Exception:
            print(f"[secrets] {name} not attached -- skipped")
    have_api = bool(os.environ.get("KAGGLE_API_TOKEN"))
except ImportError:
    print("[secrets] not running on Kaggle -- using environment as-is")
    have_api = bool(os.environ.get("KAGGLE_API_TOKEN"))
print("Kaggle API available:", have_api)

In [ ]:
# ---- Clone the repo at GIT_REF ----------------------------------------------
if os.path.exists(SRC_DIR):
    shutil.rmtree(SRC_DIR)
r1 = sh(["git", "clone", "--quiet", REPO_URL, SRC_DIR])
r2 = sh(["git", "-C", SRC_DIR, "checkout", "--quiet", GIT_REF]) if r1.returncode == 0 else r1
status["clone"] = "OK" if r2.returncode == 0 else f"FAIL: git ({GIT_REF})"
if status["clone"] == "OK":
    sh(["git", "-C", SRC_DIR, "log", "--oneline", "-3"])
print("clone:", status["clone"])

In [ ]:
# ---- Detect GPU architecture and build --------------------------------------
gpu = sh("nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader")
sm_arch = "sm_75"                    # T4 default (P100 is retired on Kaggle)
if gpu.returncode == 0 and gpu.stdout.strip():
    name, cap = [s.strip() for s in gpu.stdout.strip().splitlines()[0].split(",")]
    sm_arch = "sm_" + cap.replace(".", "")
    print(f"GPU: {name}  ->  {sm_arch}")
else:
    print("No GPU visible -- enable a GPU accelerator for this notebook")

if status["clone"] == "OK":
    b = sh(f"make -C {SRC_DIR} SM_ARCH={sm_arch} -j4")
    ok = b.returncode == 0 and os.path.exists(f"{SRC_DIR}/ga_stomp")
    status["build"] = "OK" if ok else "FAIL: nvcc"
    with open(f"{OUT_DIR}/build.log", "w") as f:
        f.write(b.stdout + "\n" + b.stderr)
else:
    status["build"] = "SKIPPED: no checkout"
print("build:", status["build"])

In [ ]:
# ---- Locate (or download) CWRU and convert .mat -> CSV ----------------------
import numpy as np, scipy.io

def find_mat(fname):
    hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True) + \
           glob.glob(f"{DATA_DIR}/cwru/**/{fname}", recursive=True)
    return hits[0] if hits else None

if find_mat("105.mat") is None and have_api:
    print("CWRU not attached -- downloading via Kaggle API")
    sh(["kaggle", "datasets", "download", "-d", CWRU_SLUG,
        "-p", f"{DATA_DIR}/cwru", "--unzip", "-q"])

csv_paths = {}
for name, (mat, _) in DATASETS.items():
    path = find_mat(mat)
    if path is None:
        status[f"data:{name}"] = f"FAIL: {mat} not found (attach {CWRU_SLUG} or add KAGGLE_API_TOKEN)"
        continue
    m = scipy.io.loadmat(path)
    key = next((k for k in m if k.endswith("DE_time")), None)
    if key is None:
        status[f"data:{name}"] = f"FAIL: no DE_time key in {mat}"
        continue
    sig = m[key].flatten()[:N_SAMPLES]
    out = f"{DATA_DIR}/{name}.csv"
    np.savetxt(out, sig, fmt="%.8f")
    csv_paths[name] = out
    status[f"data:{name}"] = "OK"
    print(f"{mat:>8} -> {out}  key={key}  n={len(sig)}")

In [ ]:
# ---- Run GA-STOMP on each signal --------------------------------------------
for name, (_, expected) in DATASETS.items():
    if status.get("build") != "OK" or name not in csv_paths:
        status[f"run:{name}"] = "SKIPPED"
        continue
    out_dir = f"{OUT_DIR}/{name}"
    os.makedirs(out_dir, exist_ok=True)
    cfg_path = f"{out_dir}/input_config.ini"
    with open(cfg_path, "w") as f:
        f.write(CONFIG_TEMPLATE.format(input_path=csv_paths[name], output_dir=out_dir))

    print("=" * 64, f"\n{name}  (expected m = {expected})\n" + "=" * 64)
    r = subprocess.run([f"{SRC_DIR}/ga_stomp", cfg_path], text=True, capture_output=True)
    with open(f"{out_dir}/stdout.log", "w") as f:
        f.write(r.stdout + ("\nSTDERR:\n" + r.stderr if r.stderr else ""))
    for line in r.stdout.splitlines():
        if line.startswith("[fft]") or "Best window_size" in line or "Composite fitness" in line:
            print(line)
    ok = r.returncode == 0 and os.path.exists(f"{out_dir}/summary.csv")
    status[f"run:{name}"] = "OK" if ok else f"FAIL: exit {r.returncode}"
    if not ok:
        print(r.stdout[-2000:], r.stderr[-2000:])

In [ ]:
# ---- Figures, summary table, status file ------------------------------------
run_dirs = [f"{OUT_DIR}/{n}" for n in DATASETS if status.get(f"run:{n}") == "OK"]
if run_dirs and os.path.exists(f"{SRC_DIR}/plot_results.py"):
    expected = [f"{n}={e}" for n, (_, e) in DATASETS.items() if e]
    p = sh(["python", f"{SRC_DIR}/plot_results.py", *run_dirs,
            "--expected", *expected, "--out", f"{OUT_DIR}/figures"])
    status["plots"] = "OK" if p.returncode == 0 else "FAIL: plot_results.py"

summary = f"{OUT_DIR}/figures/combined_summary.csv"
if os.path.exists(summary):
    import pandas as pd
    df = pd.read_csv(summary)
    try:
        display(df)
    except NameError:
        print(df.to_string())

with open(f"{OUT_DIR}/RUN_STATUS.txt", "w") as f:
    f.write(f"git_ref={GIT_REF}\n")
    for k, v in status.items():
        f.write(f"{k}={v}\n")
print(open(f"{OUT_DIR}/RUN_STATUS.txt").read())